# CFPB Credit Card Complaints — Exploratory Data Analysis

## Goal
Analyse consumer credit card complaints from the CFPB dataset to identify
patterns that predict whether a dispute is resolved in favour of the consumer or not.

## Key Questions
1. How imbalanced is the dispute outcome label? — defines training strategy and threshold calibration
2. Which complaint types and sub-products carry the most predictive signal for dispute outcome?
3. Are there temporal patterns in complaint volume or resolution rates worth engineering as features?
4. What is the missing data profile across key columns? — informs preprocessing pipeline design
5. What is the cardinality of categorical features (company, product, issue)? — affects encoding strategy and memory footprint
6. How does response time (date received → date sent) correlate with dispute outcome? — latency as a domain feature
7. What are the memory and I/O characteristics of loading and processing this dataset? — benchmark DataFrame size, dtypes footprint, and read time as a baseline for pipeline design

## Dataset
Source: US Consumer Finance Complaints — CFPB via Kaggle
Download: https://www.kaggle.com/datasets/kaggle/us-consumer-finance-complaints

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
file_name= "../data/consumer_complaints.csv"
df = pd.read_csv(file_name, low_memory=False)

  date_received                  product  \
0    08/30/2013                 Mortgage   
1    08/30/2013                 Mortgage   
2    08/30/2013         Credit reporting   
3    08/30/2013             Student loan   
4    08/30/2013          Debt collection   
5    08/30/2013              Credit card   
6    08/30/2013              Credit card   
7    08/30/2013  Bank account or service   
8    08/30/2013  Bank account or service   
9    09/17/2013                 Mortgage   

                              sub_product  \
0                          Other mortgage   
1                          Other mortgage   
2                                     NaN   
3                Non-federal student loan   
4                             Credit card   
5                                     NaN   
6                                     NaN   
7                        Checking account   
8                        Checking account   
9  Conventional adjustable mortgage (ARM)   

                   

### Initial inspection
Let's inspect the shape to see how many columns and rows we have and the columns details

In [13]:
df.shape
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 555957 entries, 0 to 555956
Data columns (total 18 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   date_received                 555957 non-null  str  
 1   product                       555957 non-null  str  
 2   sub_product                   397635 non-null  str  
 3   issue                         555957 non-null  str  
 4   sub_issue                     212622 non-null  str  
 5   consumer_complaint_narrative  66806 non-null   str  
 6   company_public_response       85124 non-null   str  
 7   company                       555957 non-null  str  
 8   state                         551070 non-null  str  
 9   zipcode                       551452 non-null  str  
 10  tags                          77959 non-null   str  
 11  consumer_consent_provided     123458 non-null  str  
 12  submitted_via                 555957 non-null  str  
 13  date_sent_to_company     

> **Data structure Note:** Analyse the type of products to filter by Credit cards

In [12]:
df['product'].value_counts()

product
Mortgage                   186475
Debt collection            101052
Credit reporting            91854
Credit card                 66468
Bank account or service     62563
Consumer Loan               20990
Student loan                15839
Payday loan                  3877
Money transfers              3812
Prepaid card                 2470
Other financial service       557
Name: count, dtype: int64

### Filter: Credit Card complaints only

> **Infrastructure Note:** filtering in-place (`df = df[...]`) rather than using `.copy()` drops the reference to the full 555k-row DataFrame, allowing the garbage collector to reclaim ~560 MB. At this scale the difference is minor, but on GB-sized datasets avoiding unnecessary copies is a first-class concern — peak memory during a copy can be 2x the DataFrame size.

In [22]:
df = df[df['product'] == 'Credit card']
print(df.shape)

(66468, 12)


In [23]:
df['consumer_disputed?'].value_counts(normalize=True)

consumer_disputed?
No     0.792171
Yes    0.207829
Name: proportion, dtype: float64

### Target Variable Analysis — `consumer_disputed?`

> **Problem type:** Binary classification — `consumer_disputed?` is either `"Yes"` (consumer opened a dispute) or `"No"` (didn't dispute).

> **Class imbalance:** 79.2% No / 20.8% Yes (~4:1 ratio). The dataset is imbalanced but not extreme.

> **Accuracy trap:** A model that always predicts "No" achieves 79% accuracy without learning anything — it has 0% recall on the minority class and never catches a real dispute. Accuracy is a misleading metric here.

In [26]:
df.isnull().sum().sort_values()

date_received                     0
product                           0
issue                             0
company                           0
submitted_via                     0
date_sent_to_company              0
company_response_to_consumer      0
timely_response                   0
consumer_disputed?                0
complaint_id                      0
zipcode                         585
state                           593
dtype: int64

### Missing Data Analysis

> **Dropped columns (>80% null):** `sub_issue`, `sub_product`, `consumer_complaint_narrative`, `company_public_response`, `tags`, `consumer_consent_provided` — these fields were not captured for credit card complaints and carry no signal.

> **Remaining nulls:** `state` (593) and `zipcode` (585) — ~0.9% of rows. Geographic data is missing for a small fraction of complaints.

> **Imputation decision:** Mean imputation is not applicable to categorical/geographic columns. Options are: most-frequent value (introduces false signal), row removal (loses valid complaint data), or placeholder `"Unknown"` (honest — treats missing geography as its own category). Decision is to use "Unknown"

In [ ]:
df['state'] = df['state'].fillna('Unknown')
df['zipcode'] = df['zipcode'].fillna('Unknown')
print(df.isnull().sum())

In [28]:
df.drop(columns=['consumer_consent_provided', 'tags', 'company_public_response', 'consumer_complaint_narrative', 'sub_issue', 'sub_product'], inplace=True, errors='ignore')
print(df.shape)

(66468, 12)
